# Gold Layer: Dimensional Modeling

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS tpch_dev.gold

In [0]:
%sql
CREATE OR REPLACE TABLE tpch_dev.gold.dim_geography AS
SELECT
  n.nation_key  AS geography_key,
  n.nation_key,
  n.nation_name,
  n.nation_comment,
  n.region_key,
  r.region_name,
  r.region_comment
FROM tpch_dev.silver.nation   n
JOIN tpch_dev.silver.region   r ON n.region_key = r.region_key

In [0]:
%sql
CREATE OR REPLACE TABLE tpch_dev.gold.dim_customer AS
SELECT
  c.customer_key,
  c.customer_name,
  c.customer_address,
  c.customer_phone,
  c.customer_acctbal,
  c.customer_mktsegment,
  c.customer_comment,
  c.nation_key        AS customer_nation_key,
  g.nation_name      AS customer_nation_name,
  g.region_key       AS customer_region_key,
  g.region_name      AS customer_region_name
FROM tpch_dev.silver.customer       c
LEFT JOIN tpch_dev.gold.dim_geography g ON c.nation_key = g.geography_key

In [0]:
%sql
CREATE OR REPLACE TABLE tpch_dev.gold.dim_product AS
SELECT
  p.part_key        AS product_key,
  p.part_name       AS product_name,
  p.part_mfgr       AS manufacturer,
  p.part_brand      AS brand,
  p.part_type       AS product_type,
  p.part_size       AS size,
  p.part_container  AS container,
  p.part_retailprice AS retail_price,
  p.part_comment    AS product_comment
FROM tpch_dev.silver.part p

In [0]:
%sql
CREATE OR REPLACE TABLE tpch_dev.gold.dim_supplier AS
SELECT
  s.supplier_key,
  s.supplier_name,
  s.supplier_address,
  s.supplier_phone,
  s.supplier_acctbal,
  s.supplier_comment,
  s.nation_key      AS supplier_nation_key,
  g.nation_name    AS supplier_nation_name,
  g.region_key      AS supplier_region_key,
  g.region_name    AS supplier_region_name
FROM tpch_dev.silver.supplier       s
LEFT JOIN tpch_dev.gold.dim_geography g ON s.nation_key = g.geography_key

In [0]:
%sql
CREATE OR REPLACE TABLE tpch_dev.gold.fact_sales AS
SELECT
  li.order_key,
  li.line_number,
  o.customer_key,
  li.part_key        AS product_key,
  li.supplier_key,
  o.order_date,
  o.order_status,
  o.order_priority,
  o.clerk            AS order_clerk,
  li.quantity,
  li.extended_price  AS gross_amount,
  li.discount,
  li.tax,
  li.extended_price * li.discount                  AS discount_amount,
  li.extended_price * (1 - li.discount)            AS discounted_amount,
  li.extended_price * (1 - li.discount) * li.tax   AS tax_amount,
  li.extended_price * (1 - li.discount) * (1 + li.tax) AS net_revenue,
  li.return_flag,
  li.line_status,
  li.ship_date,
  li.commit_date,
  li.receipt_date,
  li.ship_mode,
  li.ship_instruct,
  li.line_comment,
  -- Customer geography (from orders -> customer -> geography)
  dc.customer_nation_key,
  dc.customer_nation_name,
  dc.customer_region_key,
  dc.customer_region_name,
  -- Supplier geography (from lineitem -> supplier -> geography)
  ds.supplier_nation_key,
  ds.supplier_nation_name,
  ds.supplier_region_key,
  ds.supplier_region_name
FROM tpch_dev.silver.lineitem li
JOIN tpch_dev.silver.orders    o  ON li.order_key    = o.order_key
LEFT JOIN tpch_dev.gold.dim_customer  dc ON o.customer_key = dc.customer_key
LEFT JOIN tpch_dev.gold.dim_supplier  ds ON li.supplier_key = ds.supplier_key

In [0]:
%sql
CREATE OR REPLACE VIEW tpch_dev.gold.vw_annual_sales_kpis AS
SELECT
  YEAR(order_date)                          AS sales_year,
  COUNT(DISTINCT order_key)                 AS total_orders,
  COUNT(*)                                   AS total_line_items,
  ROUND(SUM(net_revenue), 2)                 AS total_revenue,
  COUNT(DISTINCT customer_key)               AS active_customers,
  ROUND(SUM(net_revenue) / COUNT(DISTINCT order_key), 2) AS avg_order_value
FROM tpch_dev.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY sales_year

In [0]:
%sql
WITH gold_stats AS (
  SELECT
    COUNT(*)                        AS gold_row_count,
    ROUND(SUM(net_revenue), 2)       AS gold_revenue
  FROM tpch_dev.gold.fact_sales
),
silver_stats AS (
  SELECT
    COUNT(*)                                                AS silver_row_count,
    ROUND(SUM(extended_price * (1 - discount) * (1 + tax)), 2) AS silver_revenue
  FROM tpch_dev.silver.lineitem
)
SELECT
  'Gold vs Silver Reconciliation'        AS reconciliation,
  g.gold_row_count,
  s.silver_row_count,
  g.gold_row_count - s.silver_row_count   AS row_count_diff,
  g.gold_revenue,
  s.silver_revenue,
  ROUND(g.gold_revenue - s.silver_revenue, 2) AS revenue_diff,
  CASE
    WHEN g.gold_row_count = s.silver_row_count
     AND ABS(g.gold_revenue - s.silver_revenue) < 0.01
    THEN 'PASS'
    ELSE 'FAIL'
  END                                    AS status
FROM gold_stats g
CROSS JOIN silver_stats s

In [0]:
%sql
SELECT 'dim_geography' AS dimension, COUNT(*) AS total_rows, COUNT(DISTINCT geography_key) AS distinct_keys,
  CASE WHEN COUNT(*) = COUNT(DISTINCT geography_key) THEN 'PASS' ELSE 'FAIL' END AS uniqueness_status
FROM tpch_dev.gold.dim_geography
UNION ALL
SELECT 'dim_customer', COUNT(*), COUNT(DISTINCT customer_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT customer_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_customer
UNION ALL
SELECT 'dim_product', COUNT(*), COUNT(DISTINCT product_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT product_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_product
UNION ALL
SELECT 'dim_supplier', COUNT(*), COUNT(DISTINCT supplier_key),
  CASE WHEN COUNT(*) = COUNT(DISTINCT supplier_key) THEN 'PASS' ELSE 'FAIL' END
FROM tpch_dev.gold.dim_supplier
ORDER BY dimension